In [ ]:
import os
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_parquet("Processed_data/0_nem_duid_mapping.parquet")
df["Classification"]       = df["Classification"].astype(str).str.strip()
df["Station Name"]         = df["Station Name"].astype(str).str.strip()
df["Participant"]          = df["Participant"].astype(str).str.strip()
df["Fuel Source - Primary"] = df["Fuel Source - Primary"].astype(str).str.strip()

LEVEL_COLOURS = {
    "P": "rgba(255,127,14,0.8)",
    "S": "rgba(44,160,44,0.8)",
    "D": "rgba(214,39,40,0.8)",
    "C": "rgba(148,103,189,0.8)",
    "F": "rgba(140,86,75,0.8)",
}
HIDE_LABELS = {"P", "S", "D"}

REGIONS = sorted(df["Region"].dropna().unique())

def build_sankey(region_df):
    # Order participants by descending DUID count
    participant_order = (
        region_df.groupby("Participant")["DUID"]
        .nunique()
        .sort_values(ascending=False)
        .index.tolist()
    )
    participant_rank = {p: i for i, p in enumerate(participant_order)}

    levels = [
        ("Participant",          "P"),
        ("Station Name",         "S"),
        ("DUID",                 "D"),
        ("Classification",       "C"),
        ("Fuel Source - Primary","F"),
    ]

    # Build node list; sort Participant nodes by rank
    all_nodes = []
    for col, pfx in levels:
        if col == "Participant":
            vals = sorted(region_df[col].dropna().unique(), key=lambda v: participant_rank.get(v, 999))
        else:
            vals = region_df[col].dropna().unique()
        for v in vals:
            node = f"{pfx}:{v}"
            if node not in all_nodes:
                all_nodes.append(node)

    node_idx = {n: i for i, n in enumerate(all_nodes)}

    sources, targets, values = [], [], []
    for (col_a, pfx_a), (col_b, pfx_b) in zip(levels[:-1], levels[1:]):
        for (a, b), grp in region_df.groupby([col_a, col_b]):
            src = node_idx.get(f"{pfx_a}:{a}")
            tgt = node_idx.get(f"{pfx_b}:{b}")
            if src is not None and tgt is not None:
                sources.append(src)
                targets.append(tgt)
                values.append(len(grp))

    node_colours   = [LEVEL_COLOURS[n.split(":")[0]] for n in all_nodes]
    display_labels = ["" if n.split(":")[0] in HIDE_LABELS else n.split(":", 1)[1] for n in all_nodes]
    hover_labels   = [n.split(":", 1)[1] for n in all_nodes]

    return dict(
        node_colours=node_colours,
        display_labels=display_labels,
        hover_labels=hover_labels,
        sources=sources,
        targets=targets,
        values=values,
    )


for region in REGIONS:
    region_df = df[df["Region"] == region].copy()
    s = build_sankey(region_df)

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=12,
            thickness=16,
            label=s["display_labels"],
            customdata=s["hover_labels"],
            color=s["node_colours"],
            hovertemplate="%{customdata}<br>Total flow: %{value}<extra></extra>",
        ),
        link=dict(
            source=s["sources"],
            target=s["targets"],
            value=s["values"],
        ),
    ))

    n_duids = region_df["DUID"].nunique()
    fig.update_layout(
        title_text=f"{region} — Participant → Station → DUID → Classification → Fuel Source  ({n_duids} DUIDs)",
        font_size=9,
        height=900,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    fig.show()
